# Bếp Finance — nạp snapshot Gold đã duyệt
Mẫu cho Fabric Spark notebook. Chưa chạy trên tenant. Gắn default lakehouse; upload manifest và 3 CSV theo README. Không xử lý PDF/OCR ở notebook này.

In [ ]:
snapshot_id = "REPLACE_WITH_32_CHARACTER_SNAPSHOT_ID"


In [ ]:
import json, re
from pathlib import Path
from pyspark.sql import functions as F
assert re.fullmatch(r"[0-9a-f]{32}",snapshot_id), "Set snapshot_id first"
root=Path('/lakehouse/default/Files/bep/approved_gold')/snapshot_id
manifest=json.loads((root/'manifest.json').read_text())
assert manifest['snapshot_id']==snapshot_id and manifest['currency']=='VND'
schemas={
 'fact_monthly_pnl':'branch_id STRING, month STRING, revenue_vnd LONG, food_vnd LONG, labor_vnd LONG, rent_vnd LONG, other_vnd LONG, operating_profit_vnd LONG, total_cost_vnd LONG, operating_margin_pct DOUBLE, rent_to_revenue_pct DOUBLE, month_start DATE',
 'dim_branch':'id STRING, name STRING, currency STRING',
 'dim_month':'month_start DATE, year INT, month_number INT'}
frames={name:spark.read.option('header',True).option('mode','FAILFAST').schema(schema).csv(str(root/(name+'.csv'))) for name,schema in schemas.items()}
fact=frames['fact_monthly_pnl']; branch=frames['dim_branch']; months=frames['dim_month']
assert fact.count()==manifest['fact_rows'] and fact.count()>0
for name,keys in [('fact_monthly_pnl',['branch_id','month_start']),('dim_branch',['id']),('dim_month',['month_start'])]:
 df=frames[name]
 assert df.select(keys).distinct().count()==df.count(), 'Duplicate keys: '+name
 for key in keys: assert df.filter(F.col(key).isNull()).count()==0
assert fact.join(branch,fact.branch_id==branch.id,'left_anti').count()==0
assert fact.join(months,'month_start','left_anti').count()==0
for key in ['revenue_vnd','food_vnd','labor_vnd','rent_vnd','other_vnd','total_cost_vnd','operating_profit_vnd']:
 assert fact.filter(F.col(key).isNull()).count()==0
for key in ['revenue_vnd','food_vnd','labor_vnd','rent_vnd','other_vnd']:
 assert fact.filter(F.col(key)<0).count()==0
assert fact.filter(F.col('total_cost_vnd')!=(F.col('food_vnd')+F.col('labor_vnd')+F.col('rent_vnd')+F.col('other_vnd'))).count()==0
assert fact.filter(F.col('operating_profit_vnd')!=F.col('revenue_vnd')-F.col('total_cost_vnd')).count()==0
assert branch.filter(F.col('currency')!='VND').count()==0
assert fact.filter(F.col('month')!=F.date_format('month_start','yyyy-MM')).count()==0
for numerator,column in [('operating_profit_vnd','operating_margin_pct'),('rent_vnd','rent_to_revenue_pct')]:
 expected=F.when(F.col('revenue_vnd')!=0,F.col(numerator)/F.col('revenue_vnd')*100)
 assert fact.filter(F.col(column).isNull()!=expected.isNull()).count()==0
 assert fact.filter(F.abs(F.col(column)-expected)>0.00001).count()==0
print('Validated snapshot', snapshot_id)


In [ ]:
# Versioned tables keep existing model untouched during a failed partial publication.
tables={}
for name,frame in frames.items():
 table='bep_'+name+'_'+snapshot_id
 if spark.catalog.tableExists(table):
  existing=spark.table(table).select(frame.columns)
  assert existing.exceptAll(frame).limit(1).count()==0 and frame.exceptAll(existing).limit(1).count()==0, 'Existing snapshot differs; investigate'
 else:
  frame.write.format('delta').mode('errorifexists').saveAsTable(table)
 tables[name]=table
# Ready marker is an orchestration signal, not an atomic cross-table transaction.
ready=root/'ready.json'
ready.write_text(json.dumps({'snapshot_id':snapshot_id,'tables':tables,'status':'ready_for_semantic_model_configuration'},indent=2))
print(tables)
